# Industry Employment Growth (2016 → 2025)

Which industries grew employment most during the inflationary period?

Datasets: #1 employment by industry, #2 average prices (used as an inflation benchmark).

Industries and their seasonally-adjusted CES series:

| Industry | Series ID |
|---|---|
| Construction | CES2000000001 |
| Manufacturing | CES3000000001 |
| Retail | CES4200000001 |
| Healthcare | CES6562000001 |
| Leisure & Hospitality | CES7000000001 |
| Professional Services | CES6054000001 |

In [ ]:
# plotting and data-wrangling libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## Employment (Dataset #1)

Reshape to long form. Values are in thousands; `CES` = seasonally adjusted.

In [ ]:
earn = pd.read_csv('Dataset #1 - earnings.txt')
earn = earn.rename(columns={'Series ID': 'series_id'})

# wide (one column per month) -> long (one row per series-month)
earn_long = earn.melt(id_vars=['series_id'], var_name='month', value_name='employment')
earn_long['month'] = pd.to_datetime(earn_long['month'], format='%b %Y', errors='coerce')
# strip the '(P)' preliminary tag and force numeric; blanks/future months become NaN
earn_long['employment'] = pd.to_numeric(
    earn_long['employment'].astype(str).str.replace(r'\(P\)', '', regex=True).str.strip(),
    errors='coerce')
earn_long = earn_long.dropna(subset=['month', 'employment'])
earn_long['adjusted'] = earn_long['series_id'].str.startswith('CES')

print('rows:', len(earn_long))
print('date range:', earn_long['month'].min().date(), '->', earn_long['month'].max().date())
earn_long.head()

## Inflation (Dataset #2)

Same reshape, used later as an economy-wide price benchmark. Reused from `eda.ipynb`.

In [ ]:
# map each series ID to a human-readable item name
labels = {}
with open('inflation_ids.txt') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        label, code = line.rsplit(' - ', 1)
        labels[code.strip()] = label.strip()

inf = pd.read_csv('Dataset #2 - inflation.txt')
inf = inf.rename(columns={'Series ID': 'series_id'})
inf['item'] = inf['series_id'].map(labels)

# wide -> long; coerce placeholders/blanks to NaN and drop them
inf_long = inf.melt(id_vars=['series_id', 'item'], var_name='month', value_name='price')
inf_long['month'] = pd.to_datetime(inf_long['month'], format='%b %Y', errors='coerce')
inf_long['price'] = pd.to_numeric(inf_long['price'], errors='coerce')
inf_long = inf_long.dropna(subset=['month', 'price'])

print('rows:', len(inf_long), '| items:', inf_long['item'].nunique())
inf_long.head()

## Pick the six industries

In [ ]:
# series ID -> industry label
industries = {
    'CES2000000001': 'Construction',
    'CES3000000001': 'Manufacturing',
    'CES4200000001': 'Retail',
    'CES6562000001': 'Healthcare',
    'CES7000000001': 'Leisure & Hospitality',
    'CES6054000001': 'Professional Services',
}

# keep only those series and tag each with its label and year
ind = earn_long[earn_long['series_id'].isin(industries)].copy()
ind['industry'] = ind['series_id'].map(industries)
ind['year'] = ind['month'].dt.year

# confirm all six are present, then show each industry's employment range (thousands)
print('industries found:', sorted(ind['industry'].unique()))
ind.groupby('industry')['employment'].agg(['min', 'max'])

## Percent growth, 2016 → 2025

Compare calendar-year average employment in 2025 vs 2016 to smooth seasonality. The inflation benchmark applies the same calculation to each item's price and averages across the basket.

In [ ]:
# employment: yearly mean per industry, then 2025 vs 2016 percent change
annual = ind.groupby(['industry', 'year'])['employment'].mean().reset_index()
emp_2016 = annual[annual['year'] == 2016].set_index('industry')['employment']
emp_2025 = annual[annual['year'] == 2025].set_index('industry')['employment']
emp_growth = (100 * (emp_2025 / emp_2016 - 1)).rename('employment_growth_pct')

# inflation benchmark: same percent change per item, averaged across the basket
inf_y = inf_long.copy()
inf_y['year'] = inf_y['month'].dt.year
item_annual = inf_y.groupby(['item', 'year'])['price'].mean()
price_2016 = item_annual.xs(2016, level='year')
price_2025 = item_annual.xs(2025, level='year')
item_growth = 100 * (price_2025 / price_2016 - 1)
inflation_benchmark = item_growth.mean()

# ranked table, with gap vs inflation in percentage points
summary = emp_growth.sort_values(ascending=False).round(1).to_frame()
summary['vs_inflation_pp'] = (summary['employment_growth_pct'] - inflation_benchmark).round(1)
print(f'Consumer-price inflation benchmark (basket avg, 2016->2025): {inflation_benchmark:.1f}%')
summary

## Grouped bar chart

Industries sorted by employment growth, each paired against the inflation benchmark.

In [ ]:
# sort industries by growth (largest first)
order = emp_growth.sort_values(ascending=False).index
vals = emp_growth[order].values
x = np.arange(len(order))
w = 0.38  # bar width; offset the two bars left/right of each tick

fig, ax = plt.subplots(figsize=(11, 6))
# blue: per-industry employment growth
ax.bar(x - w / 2, vals, w, label='Employment growth (2016→2025)', color='#2c7fb8')
# orange: same inflation benchmark repeated for each industry
ax.bar(x + w / 2, [inflation_benchmark] * len(order), w,
       label=f'Consumer-price inflation, basket avg ({inflation_benchmark:.0f}%)', color='#d95f0e')

ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(order, rotation=20, ha='right')
ax.set_ylabel('% Growth (2016 → 2025)')
ax.set_title('Industry Employment Growth vs. Inflation (2016 → 2025)')
ax.legend()

# label each employment bar with its value
for xi, v in zip(x, vals):
    ax.text(xi - w / 2, v + (0.6 if v >= 0 else -0.6), f'{v:.0f}%',
            ha='center', va='bottom' if v >= 0 else 'top', fontsize=9)

fig.tight_layout()
plt.show()

## Takeaway

Employment growth, 2016 → 2025:

| Industry | Growth |
|---|---|
| Construction | +22.8% |
| Healthcare | +22.1% |
| Professional Services | +21.4% |
| Leisure & Hospitality | +7.7% |
| Manufacturing | +2.5% |
| Retail | −2.2% |